# SLAM Experiments: GTSAM vs Raw Trigonometry

This notebook compares the accuracy of GTSAM-based optimization versus simple raw trigonometry for landmark localization using bearing and range measurements from a drone-mounted gimbal.

## Experiment Setup

- **Landmark**: Fixed at position [10, 0, 5] meters
- **Drone Trajectory**: Three poses moving along the y-axis (0m, 10m, 20m) with identity orientation
- **Measurements**: Gimbal pitch/yaw angles and range distance to landmark
- **Noise**: 
  - Angles: 1° standard deviation on pitch and yaw
  - Range: Variable standard deviation (0.1m, 0.5m, 1.0m, 2.0m, 10.0m)
- **Simulations**: 100 runs per noise level

## Methods Compared

1. **GTSAM Optimization**: Uses the same `run_gtsam_optimization` function as implemented in the ROS2 drone control system. This function employs GTSAM's BearingRangeFactor3D to jointly optimize landmark position across all measurements, accounting for measurement uncertainties.

2. **Raw Trigonometry**: Deterministic calculation using only the first measurement (single ray-casting approach), equivalent to the initial guess in GTSAM.

## Key Findings

The experiment demonstrates that GTSAM's joint optimization significantly outperforms single-measurement trigonometry, especially at higher noise levels, by effectively fusing multiple uncertain measurements.

In [22]:
import gtsam
import numpy as np
import math
import plotly 

from gtsam.symbol_shorthand import L, X

In [23]:
def run_gtsam_optimization(self, data_points, range_noise_std=0.1):
    # --- A. Setup Graph and Noise Models ---
    graph = gtsam.NonlinearFactorGraph()
    
    # Noise for Drone Position (Prior) - We trust MAVROS/Sim state quite a bit
    # [Roll, Pitch, Yaw, X, Y, Z] errors
    prior_noise = gtsam.noiseModel.Diagonal.Sigmas(
        np.array([0.01, 0.01, 0.01, 0.05, 0.05, 0.05])
    )
    
    # Noise for Measurements (Bearing + Range)
    # Bearing: 1 degree std (~0.01745 rad), Range: as specified
    measurement_noise = gtsam.noiseModel.Diagonal.Sigmas(
        np.array([0.01745, 0.01745, range_noise_std])
    )

    initial_estimates = gtsam.Values()
    
    # We need a key for the single landmark (Cube)
    LANDMARK_KEY = L(0)

    # --- B. Loop through Data ---
    for i, point in enumerate(data_points):
        POSE_KEY = X(i)
        
        # Extract Drone Pose
        pos = point['drone_pos'] # [x, y, z]
        quat = point['drone_quat'] # [w, x, y, z]
        
        # Construct GTSAM Pose3
        # Note: GTSAM Python Rot3.Quaternion takes (w, x, y, z)
        rot = gtsam.Rot3.Quaternion(quat[0], quat[1], quat[2], quat[3])
        pose = gtsam.Pose3(rot, gtsam.Point3(pos[0], pos[1], pos[2]))
        
        # Add Prior Factor (We trust where the drone says it is)
        graph.add(gtsam.PriorFactorPose3(POSE_KEY, pose, prior_noise))
        initial_estimates.insert(POSE_KEY, pose)
        
        # Extract Measurement
        pitch_deg = point['gimbal_pitch']
        yaw_deg = point['gimbal_yaw']
        r = point['range']
        
        # Convert Angles to Unit3 Vector (Bearing)
        # This must match the geometry that worked in your manual test
        # PITCH: Positive = Down => Z is negative
        p_rad = math.radians(pitch_deg)
        y_rad = math.radians(yaw_deg)
        
        z_b = -1.0 * math.sin(p_rad)
        xy_dist = math.cos(p_rad)
        x_b = xy_dist * math.cos(y_rad)
        y_b = xy_dist * math.sin(y_rad)
        
        bearing_vector = gtsam.Point3(x_b, y_b, z_b)
        bearing_unit = gtsam.Unit3(bearing_vector)
        
        # Add BearingRange Factor
        # Connects Drone Pose X(i) -> Landmark L(0)
        graph.add(gtsam.BearingRangeFactor3D(
            POSE_KEY, LANDMARK_KEY, bearing_unit, float(r), measurement_noise
        ))

    # --- C. Initialize Landmark Guess ---
    # Optimization needs a starting point. We use the first measurement to triangulate a guess.
    first_p = data_points[0]
    guess_pose = initial_estimates.atPose3(X(0))

    # Get reference GPS from the first data point
    ref_gps = data_points[0]['gps_ref']
    ref_pos = data_points[0]['drone_pos'] # The local (x,y) corresponding to that GPS
    
    # Calculate vector in Body frame
    p_rad = math.radians(first_p['gimbal_pitch'])
    y_rad = math.radians(first_p['gimbal_yaw'])
    r = first_p['range']
    
    # Same math as above
    local_vec = np.array([
        r * math.cos(p_rad) * math.cos(y_rad),
        r * math.cos(p_rad) * math.sin(y_rad),
        -r * math.sin(p_rad)
    ])
    
    # Rotate to World Frame
    # Get rotation matrix from the first pose estimate
    R_matrix = guess_pose.rotation().matrix()
    world_vec = R_matrix @ local_vec
    
    # Add to Drone Position
    t_vector = guess_pose.translation() # gtsam.Point3
    guess_x = t_vector[0] + world_vec[0]
    guess_y = t_vector[1] + world_vec[1]
    guess_z = t_vector[2] + world_vec[2]
    
    #print(f"Initial Guess for Cube: [{guess_x:.2f}, {guess_y:.2f}, {guess_z:.2f}]")
    initial_estimates.insert(LANDMARK_KEY, gtsam.Point3(guess_x, guess_y, guess_z))

    # --- D. Optimize ---
    params = gtsam.LevenbergMarquardtParams()
    optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimates, params)
    result = optimizer.optimize()
    
    # --- E. Extract Result ---
    final_point = result.atPoint3(LANDMARK_KEY)

    
    # Calculate offset of Cube relative to that specific drone position
    # Cube_Global = Drone_Global + (Cube_Local - Drone_Local)
    dx = final_point[0] - ref_pos[0]
    dy = final_point[1] - ref_pos[1]
    
    # The GPS translation is only relevant to the simulation.
    #   Once we get local coordinates, gps translation is determenistic
    #cube_lat, cube_lon = self.enu_to_gps(ref_gps['lat'], ref_gps['lon'], dx, dy)
    #self.get_logger().info(f"Cube GPS: Lat {cube_lat:.7f}, Lon {cube_lon:.7f}")

    return [final_point[0], final_point[1], final_point[2]]

In [24]:
import numpy as np
import math
import plotly.graph_objects as go

def compute_gimbal_angles(drone_pos, drone_quat, target_pos):
    # Compute vector in world frame
    vec_world = np.array(target_pos) - np.array(drone_pos)
    
    # Get rotation matrix from quaternion
    rot = gtsam.Rot3.Quaternion(drone_quat[0], drone_quat[1], drone_quat[2], drone_quat[3])
    R = rot.matrix()
    
    # Transform to body frame (inverse rotation)
    vec_body = R.T @ vec_world
    
    # Compute range
    r = np.linalg.norm(vec_body)
    
    # Compute pitch and yaw
    pitch_rad = math.atan2(-vec_body[2], math.sqrt(vec_body[0]**2 + vec_body[1]**2))
    yaw_rad = math.atan2(vec_body[1], vec_body[0])
    
    pitch_deg = math.degrees(pitch_rad)
    yaw_deg = math.degrees(yaw_rad)
    
    return pitch_deg, yaw_deg, r

In [25]:
# Experiment setup
true_landmark = [10, 0, 5]

poses = [
    {"pos": [0, 0, 0], "quat": [1, 0, 0, 0]},
    {"pos": [0, 10, 0], "quat": [1, 0, 0, 0]},
    {"pos": [0, 20, 0], "quat": [1, 0, 0, 0]},
]

# Compute true measurements
true_measurements = []
for pose in poses:
    pitch, yaw, r = compute_gimbal_angles(pose["pos"], pose["quat"], true_landmark)
    true_measurements.append({"pitch": pitch, "yaw": yaw, "range": r})

noise_levels = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
num_simulations = 100

gtsam_errors = {noise: [] for noise in noise_levels}
raw_errors = {noise: [] for noise in noise_levels}

for noise in noise_levels:
    for sim in range(num_simulations):
        data_points = []
        for i, pose in enumerate(poses):
            pitch, yaw, r = true_measurements[i]["pitch"], true_measurements[i]["yaw"], true_measurements[i]["range"]
            noisy_pitch = pitch + np.random.normal(0, 1.0)  # 1 degree std noise on angles
            noisy_yaw = yaw + np.random.normal(0, 1.0)
            noisy_r = r + np.random.normal(0, noise)
            entry = {
                "drone_pos": pose["pos"],
                "drone_quat": pose["quat"],
                "gimbal_pitch": noisy_pitch,
                "gimbal_yaw": noisy_yaw,
                "range": noisy_r,
                "gps_ref": {"lat": 0, "lon": 0, "alt": 0} # we don't care about gps here
            }
            data_points.append(entry)
        
        # Run GTSAM
        gtsam_result = run_gtsam_optimization(None, data_points, noise)  # self is None since not in class
        gtsam_error = np.linalg.norm(np.array(gtsam_result) - np.array(true_landmark))
        gtsam_errors[noise].append(gtsam_error)
        
        # Raw trigonometry: initial guess from first measurement
        first_p = data_points[0]
        guess_pose = gtsam.Pose3(
            gtsam.Rot3.Quaternion(first_p['drone_quat'][0], first_p['drone_quat'][1], first_p['drone_quat'][2], first_p['drone_quat'][3]),
            gtsam.Point3(first_p['drone_pos'][0], first_p['drone_pos'][1], first_p['drone_pos'][2])
        )
        
        p_rad = math.radians(first_p['gimbal_pitch'])
        y_rad = math.radians(first_p['gimbal_yaw'])
        r = first_p['range']
        
        local_vec = np.array([
            r * math.cos(p_rad) * math.cos(y_rad),
            r * math.cos(p_rad) * math.sin(y_rad),
            -r * math.sin(p_rad)
        ])
        
        R_matrix = guess_pose.rotation().matrix()
        world_vec = R_matrix @ local_vec
        
        t_vector = guess_pose.translation()
        guess_x = t_vector[0] + world_vec[0]
        guess_y = t_vector[1] + world_vec[1]
        guess_z = t_vector[2] + world_vec[2]
        
        raw_result = [guess_x, guess_y, guess_z]
        raw_error = np.linalg.norm(raw_result - np.array(true_landmark))
        raw_errors[noise].append(raw_error)

In [ ]:
# Save Results Data
import json

results_data = {
    "noise_levels": noise_levels,
    "gtsam_errors": gtsam_errors,
    "raw_errors": raw_errors,
    "true_landmark": true_landmark,
    "poses": poses,
    "num_simulations": num_simulations
}

with open('slam_experiment_results.json', 'w') as f:
    json.dump(results_data, f, indent=2)

print("Results saved to slam_experiment_results.json")

In [26]:
# Plotting
fig = go.Figure()

for noise in noise_levels:
    fig.add_trace(go.Box(
        y=gtsam_errors[noise],
        name=f'GTSAM σ={noise}',
        boxpoints='all',
        jitter=0.3,
        pointpos=-1.8
    ))
    fig.add_trace(go.Box(
        y=raw_errors[noise],
        name=f'Raw σ={noise}',
        boxpoints='all',
        jitter=0.3,
        pointpos=-1.8
    ))

fig.update_layout(
    title='Comparison of GTSAM vs Raw Trigonometry Errors',
    yaxis_title='Error (meters)',
    xaxis_title='Method and Noise Level',
    boxmode='group'
)

fig.show()

In [ ]:
# Additional Plots
# 1. Mean Error vs Noise Level
mean_gtsam = [np.mean(gtsam_errors[noise]) for noise in noise_levels]
std_gtsam = [np.std(gtsam_errors[noise]) for noise in noise_levels]
mean_raw = [np.mean(raw_errors[noise]) for noise in noise_levels]
std_raw = [np.std(raw_errors[noise]) for noise in noise_levels]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=noise_levels,
    y=mean_gtsam,
    error_y=dict(type='data', array=std_gtsam),
    mode='lines+markers',
    name='GTSAM',
    line=dict(color='blue')
))
fig2.add_trace(go.Scatter(
    x=noise_levels,
    y=mean_raw,
    error_y=dict(type='data', array=std_raw),
    mode='lines+markers',
    name='Raw Trigonometry',
    line=dict(color='red')
))
fig2.update_layout(
    title='Mean Position Error vs Range Noise Level',
    xaxis_title='Range Noise Std (m)',
    yaxis_title='Mean Error (m)',
    yaxis_type='log'
)
fig2.show()

# 2. Error Reduction Ratio
ratios = [np.mean(raw_errors[noise]) / np.mean(gtsam_errors[noise]) for noise in noise_levels]

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=[str(n) for n in noise_levels],
    y=ratios,
    name='Error Reduction Ratio'
))
fig3.update_layout(
    title='Raw Error / GTSAM Error Ratio',
    xaxis_title='Range Noise Std (m)',
    yaxis_title='Error Ratio'
)
fig3.show()
